In [2]:
import stanza

stanza.download("fr")
nlp = stanza.Pipeline('fr')

2026-03-19 10:25:19 INFO: Downloaded file to /Users/marwan/Library/Caches/stanza/1.11.0/resources/resources.json
2026-03-19 10:25:19 INFO: Downloading default packages for language: fr (French) ...
2026-03-19 10:26:36 INFO: Downloaded file to /Users/marwan/Library/Caches/stanza/1.11.0/resources/fr/default.zip
2026-03-19 10:26:37 INFO: Finished downloading models and saved to /Users/marwan/Library/Caches/stanza/1.11.0/resources
2026-03-19 10:26:37 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-19 10:26:37 INFO: Downloaded file to /Users/marwan/Library/Caches/stanza/1.11.0/resources/resources.json
2026-03-19 10:26:37 INFO: Loading these models for language: fr (French):
| Processor | Package            |
----------------------------------
| tokenize  | combined           |
| mwt       | combined           |
| pos       | combined_charl

In [4]:
doc = nlp("François Hollande est une limace")
print(doc.entities)

[{
  "text": "François Hollande",
  "type": "PER",
  "start_char": 0,
  "end_char": 17
}]


Now on an Aioli annotation

In [18]:
from arango import ArangoClient, database

client = ArangoClient(hosts="http://localhost:8529")
db : database.StandardDatabase = client.db("TEATIME", username="root", password="test")
annotations_collection : database.StandardCollection = db.collection("aioli_objects")

annotations_list = annotations_collection.all()
annotations_list = [_ for _ in annotations_list]

In [32]:
count_general = 0
count_desc = 0
total = len(annotations_list)
for a in annotations_list:
    if 'description' in a:
        if a['description'] is not None:
            count_desc += 1
            if 'Général' in a['description']:
                count_general += 1

print(f"{count_desc} out of {total} ({round(count_desc/total * 100, 1)}%) annotations contain a description")
print(f"{count_general} out of {total} ({round(count_general/total * 100, 1)}%) annotations contain a general description")

16184 out of 18344 (88.2%) annotations contain a description
4867 out of 18344 (26.5%) annotations contain a general description


By doing it only on the general description field, we can only structure a quarter of our dataset. However, we might do it on every field of the description field, giving us a larger coverage but very variable number of Entities. The number of named entities could help us with determining the documents granularity/LoD.

In [17]:
doc = nlp(annotations_list[0]['description']['Général']) # 0 or 123

print("Texte original :")
print(doc.text)
print("Entités : ")
print(*[f'entity: {ent.text}\ttype: {ent.type}' for ent in doc.ents], sep='\n')
"""
doc = nlp(annotations_list[123]['description']['Général'])

print(doc.text)
print(*[f'entity: {ent.text}\ttype: {ent.type}' for ent in doc.ents], sep='\n')"""

Texte original :
Les travées flanquées à la croisée, n'ont pas pu être relevées en raison de la dépose, en cours, de l'échafaudage incendié et des vestiges non déblayés qui recouvrent les extrados. Un constat d'état complémentaire sera réalisé sur place lorsque l'accès sera dégagé.

. Reins : Les reins de voûtes sont remplis de plomb fondu provenant de la couverture.
. Voûtains: Pierres dégradées, fracturation ou déplaquage sur plusieurs centimètres à l'extrados des claveaux provoqué par le choc thermique de l'incendie, changement de couleurs indiquant des changements minéralogiques et de possibles modifications mécaniques. Lacunes ponctuelles: Zones affectées par la
chute de bois de charpente, la durée et la puissance de l'incendie. Zones fragilisées au droit des pierres lacunaires. Joints dégradés et évidés en profondeur, phénomène de décarbonatation des joints à la chaux. Chape de protection irrégulière et déplaquée à près de 80%, pouvant s'expliquer par la désagrégation du mortier 

"\ndoc = nlp(annotations_list[123]['description']['Général'])\n\nprint(doc.text)\nprint(*[f'entity: {ent.text}\ttype: {ent.type}' for ent in doc.ents], sep='\n')"

Il reste du travail a faire pour rendre la détection plus fiable, elle ne couvre pas entièrement notre domaine